# Threat Detection: PCA Anomaly Detection on Hybrid Hardware

This notebook implements real-time threat detection via **PCA-based anomaly scoring**
on the uniqx platform. Eigendecomposition of sensor covariance matrices identifies
principal components; anomalies are detected by large reconstruction errors.

| Workload | Key op | Hardware |
|:---------|:-------|:---------|
| PCA decomposition | `eigs` (covariance matrix) | GPU at high dimensions |
| Temporal evolution | `expv` (state-space model) | GPU / QPU |
| Real-time scoring | `matmul` + `dot` (projection) | CPU (low latency) |

We test 5 threat profiles (port scan, data exfiltration, DDoS, lateral movement,
crypto mining) across 3 sensor types, showing how feature dimension drives
the CPU/GPU hardware decision.

**All computation goes through uniqx.**

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx.domains.ml.security import (
    SensorGrid,
    THREAT_PROFILES,
    build_anomaly_detection_module,
    build_temporal_anomaly_module,
)
from uniqx import parse_result
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Threat Profiles

In [ ]:
print(f"{'Threat':<20} {'Magnitude':>10} {'Affected features'}")
print("-" * 55)
for name, profile in THREAT_PROFILES.items():
    print(
        f"{name:<20} {profile['anomaly_magnitude']:>10.1f} {profile['affected_features']}"
    )

grids = {
    "network": SensorGrid.network_traffic(n_features=16),
    "iot": SensorGrid.industrial_iot(n_features=32),
    "finance": SensorGrid.financial_transactions(n_features=24),
}
print("\nSensor grids:")
for name, g in grids.items():
    print(f"  {name}: {g.n_features} features — {g.description}")

## 2. PCA Decomposition: Preflight Comparison

In [ ]:
def print_opts(label, options):
    print(f"\n--- {label} ---")
    if not options:
        return
    for o in options:
        f = " *" if o.get("recommended") else ""
        print(
            f"  {o['label']:<20} time={o['total_time']:>8.1f}  cost=${o['total_cost_usd']:.4f}  err={o['max_error_rate']:.4f}{f}"
        )


pca_results = {}

for grid_name, grid in grids.items():
    mod, inputs, meta = build_anomaly_detection_module(
        grid, n_samples=128, threat_type="port_scan"
    )
    opts = preflight(mod, client=client)
    print_opts(f"PCA: {grid_name} ({grid.n_features} features)", opts)

    pca_results[grid_name] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["eigenvalues", "eigenvectors"])
        evals = out.get("eigenvalues", ([], "", []))[2] if out else []

        pca_results[grid_name][opt["label"]] = {
            "eigenvalues": evals,
            "time": wall,
            "option": opt,
            "labels": meta["labels"],
        }
        print(
            f"  {opt['label']}: {wall:.2f}s, top eigenvalue={evals[0]:.4f}" if evals else f"  {opt['label']}: {wall:.2f}s (no result)"
            if evals
            else f"  {opt['label']}: {wall:.2f}s"
        )

## 3. Threat Profile Comparison

In [ ]:
grid = grids["network"]
threat_evals = {}

for threat in THREAT_PROFILES:
    mod, inputs, meta = build_anomaly_detection_module(
        grid, n_samples=128, threat_type=threat
    )
    opts = preflight(mod, client=client)
    rec = opts.recommended

    t0 = time.monotonic()
    jid = submit(
        mod,
        client=client,
        runtime_inputs=inputs,
        preflight_job_id=opts.job_id,
        option_idx=rec["_idx"],
    )
    res = get(jid, client=client)
    wall = time.monotonic() - t0

    payload = res.get("payload", b"")
    if isinstance(payload, str):
        payload = payload.encode()
    out = parse_result(payload, ["eigenvalues", "eigenvectors"])
    evals = out.get("eigenvalues", ([], "", []))[2] if out else []

    threat_evals[threat] = evals
    var_explained = (evals[0] / sum(evals) * 100) if evals and sum(evals) > 0 else 0
    print(
        f"  {threat:<20}: top PC explains {var_explained:.1f}% variance ({wall:.2f}s)"
    )

## 4. Temporal Anomaly Detection (expv)

In [ ]:
temporal_results = {}

for grid_name, grid in grids.items():
    mod, inputs, meta = build_temporal_anomaly_module(grid, n_timesteps=32)
    opts = preflight(mod, client=client)
    print_opts(f"Temporal: {grid_name} ({grid.n_features} features)", opts)

    temporal_results[grid_name] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0
        temporal_results[grid_name][opt["label"]] = {"time": wall, "option": opt}
        print(f"  {opt['label']}: {wall:.2f}s")

## 5. Feature Dimension Scaling

In [ ]:
scaling = []
for n_feat in [8, 16, 32, 48]:
    grid_s = SensorGrid("scale_test", n_features=n_feat)
    mod, inputs, meta = build_anomaly_detection_module(grid_s, n_samples=128)
    opts = preflight(mod, client=client)
    print(f"\nn_features={n_feat} (cov matrix {n_feat}x{n_feat}):")
    for opt in opts:
        f = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}  cost=${opt['total_cost_usd']:.4f}{f}"
        )
        scaling.append(
            {
                "n": n_feat,
                "label": opt["label"],
                "time": opt["total_time"],
                "cost": opt["total_cost_usd"],
                "recommended": opt.get("recommended", False),
            }
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# Top-left: Eigenvalue spectra by threat type (scree plot)
for threat, evals in threat_evals.items():
    if evals:
        total = sum(abs(e) for e in evals) or 1.0
        cumvar = np.cumsum([abs(e) / total for e in evals])
        axes[0, 0].plot(cumvar, label=threat, linewidth=1.5)
axes[0, 0].axhline(0.95, color="gray", linestyle="--", alpha=0.5, label="95% threshold")
axes[0, 0].set_xlabel("Principal component")
axes[0, 0].set_ylabel("Cumulative variance explained")
axes[0, 0].set_title("Scree Plot by Threat Type")
axes[0, 0].legend(fontsize=7)
axes[0, 0].grid(alpha=0.3)

# Top-right: PCA execution time across grids
grid_names = list(pca_results.keys())
for hw in sorted(set(l for g in pca_results for l in pca_results[g])):
    times = [pca_results[g].get(hw, {}).get("time", 0) for g in grid_names]
    n_feats = [grids[g].n_features for g in grid_names]
    c = colors_hw.get(hw, "#6b7280")
    axes[0, 1].plot(n_feats, times, "o-", color=c, label=hw)
axes[0, 1].set_xlabel("Feature dimensions")
axes[0, 1].set_ylabel("Wall time (s)")
axes[0, 1].set_title("PCA: Execution Time by Grid")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# Bottom-left: Temporal vs PCA comparison
for grid_name in grid_names:
    pca_t = (
        min(d.get("time", 999) for d in pca_results[grid_name].values())
        if pca_results[grid_name]
        else 0
    )
    temp_t = (
        min(d.get("time", 999) for d in temporal_results.get(grid_name, {}).values())
        if temporal_results.get(grid_name)
        else 0
    )
    axes[1, 0].bar(
        grid_name,
        pca_t,
        color="#2563eb",
        alpha=0.7,
        label="PCA (eigs)" if grid_name == grid_names[0] else "",
    )
    axes[1, 0].bar(
        grid_name,
        temp_t,
        bottom=pca_t,
        color="#9333ea",
        alpha=0.7,
        label="Temporal (expv)" if grid_name == grid_names[0] else "",
    )
axes[1, 0].set_ylabel("Wall time (s)")
axes[1, 0].set_title("PCA vs Temporal Detection")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(axis="y", alpha=0.3)

# Bottom-right: Scaling
hw_labels = sorted(set(d["label"] for d in scaling))
for hw in hw_labels:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].plot(
        [d["n"] for d in sub], [d["time"] for d in sub], "o-", color=c, label=hw
    )
axes[1, 1].set_xlabel("Feature dimensions")
axes[1, 1].set_ylabel("Estimated time")
axes[1, 1].set_title("Hardware Scaling with Dimension")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Threat Detection on Hybrid Hardware", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Workload | Dimension | Best hardware | Latency |
|:---------|:----------|:--------------|:--------|
| PCA (16 features) | 16x16 cov | CPU | < 1s |
| PCA (32 features) | 32x32 cov | GPU crossover | ~1s |
| PCA (48 features) | 48x48 cov | GPU | ~2s |
| Temporal (expv) | n x n transition | GPU / QPU | varies |

DDoS and port scan threats create the strongest eigenvalue signatures
(highest variance in first PC). Lateral movement is subtlest (spread
across many components). The cost model routes large covariance
decompositions to GPU while keeping real-time scoring on CPU.